# Day 3.11 — Pivotal Exercise: Enforce Action Policy

This is the day's one hands-on implementation lab. It uses no API key. Try the starter cell first; a fully commented reference solution follows the check so you can compare or catch up.

## Why this mechanism matters

A model proposal is not authorization. Policy evaluates a structured action against available capabilities and approval state before any side-effecting handler runs.

## Contract

Deny unknown tools. Permit read-only tools. Permit a side-effecting tool only when its exact `action_id` is in `approved_action_ids`. Return `(allowed, reason)`.

Before coding, write one sentence predicting the easiest mistake to make.

In [ ]:
def evaluate_action(action, allowed_tools, approved_action_ids):
    """Return (allowed: bool, reason: str).

    action        -> {"action_id": "a1", "tool": "search", ...}
    allowed_tools -> {"search": {"side_effect": False}, "send_email": {"side_effect": True}}
    """
    # TODO: reject unknown tools before considering approval
    # TODO: allow read-only tools
    # TODO: require the exact action_id to be approved for side effects
    raise NotImplementedError("Complete action policy")

## Behavioural check

Run this after completing the starter cell. If you have not finished, it prints a hint instead of failing. A passing check proves the listed contract examples, not every possible input.

In [ ]:
def run_checks():
    allowed = {"search": {"side_effect": False}, "send_email": {"side_effect": True}}
    cases = [
        ({"action_id": "a1", "tool": "search"},     set(),  True,  "read-only tool runs without approval"),
        ({"action_id": "a2", "tool": "send_email"}, set(),  False, "side effect without approval is blocked"),
        ({"action_id": "a2", "tool": "send_email"}, {"a2"}, True,  "side effect with exact approval runs"),
        ({"action_id": "a2", "tool": "send_email"}, {"a9"}, False, "approval for a different action id does not transfer"),
        ({"action_id": "a3", "tool": "delete_all"}, {"a3"}, False, "unknown tool is denied even if 'approved'"),
    ]
    for action, approvals, expected, label in cases:
        allowed_flag, reason = evaluate_action(action, allowed, approvals)
        print(f"{'ALLOW' if allowed_flag else 'DENY ':5} {action['tool']:<10} approvals={sorted(approvals)!s:<8} -> {reason}")
        assert allowed_flag == expected, label
    print("PASS: capability and exact-action approval are enforced")

try:
    run_checks()
except NotImplementedError:
    print("Not implemented yet. Complete the starter cell above, or study the reference solution below and re-run this cell.")

## Reference solution

Read this even if your check passed: compare each commented line with your version, then re-run the check cell above.

In [ ]:
# --- Reference solution: read it line by line, then re-run the check cell above ---
def evaluate_action(action, allowed_tools, approved_action_ids):
    tool = action.get("tool")
    if tool not in allowed_tools:                              # 1. unknown capability -> fail closed
        return False, f"unknown tool {tool!r}"
    if not allowed_tools[tool]["side_effect"]:                 # 2. read-only -> no approval needed
        return True, "read-only tool"
    if action.get("action_id") in approved_action_ids:         # 3. side effect -> exact id must be approved
        return True, "side effect approved for this exact action id"
    return False, "side effect requires approval"              # 4. default: do not run

print("Reference evaluate_action defined. Re-run the check cell above to see PASS.")

## Explain

**Why is approving the exact structured action safer than approving a sentence such as 'send it'?**

<details><summary>Show answer</summary>

'Send it' does not say what, to whom, or with which content. If the draft or recipients change after the sentence was spoken, the approval silently covers something the person never saw. An action id binds approval to one exact payload.

</details>

**Why must the unknown-tool check come first?**

<details><summary>Show answer</summary>

Otherwise an approval set containing a stray id could authorize a tool nobody declared. Capability is checked before approval so approval can never widen the capability list.

</details>